In [1]:

from pathlib import Path
import boto3
import tarfile
import tempfile
import os

def tar_and_upload_to_s3(
    source_dir,
    bucket,
    key,
    *,
    profile=None,
    remove_local=True,
    verbose=True,
):
    source_dir = Path(source_dir).resolve()
    if not source_dir.exists():
        raise FileNotFoundError(f"source_dir not found: {source_dir}")

    # Create tarball in a temp dir
    tmpdir = Path(tempfile.mkdtemp(prefix="hf_model_tar_"))
    tar_path = tmpdir / (Path(key).name if key.endswith(".tar.gz") else f"{Path(key).name}.tar.gz")

    if verbose:
        print("Source:", source_dir)
        print("Temp tar:", tar_path)
        print("S3 target:", f"s3://{bucket}/{key}")

    # IMPORTANT: preserve symlinks (do NOT dereference)
    # tarfile default is dereference=False, but we set explicitly.
    with tarfile.open(tar_path, "w:gz", dereference=False) as tar:
        # Put the folder into the tar with its folder name as the root
        tar.add(str(source_dir), arcname=source_dir.name, recursive=True)

    size_mb = tar_path.stat().st_size / (1024**2)
    if verbose:
        print(f"Tar created: {size_mb:.2f} MB")

    # Upload
    session = boto3.Session(profile_name=profile) if profile else boto3.Session()
    s3 = session.client("s3")
    s3.upload_file(str(tar_path), bucket, key)

    if verbose:
        print("Upload complete")

    # Cleanup
    if remove_local:
        try:
            tar_path.unlink(missing_ok=True)
            tmpdir.rmdir()
        except Exception:
            pass

    return str(tar_path)  # returns path if you set remove_local=False



In [ ]:
LLAMA_DIR = "/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store"

tar_and_upload_to_s3(
    source_dir=LLAMA_DIR,
    bucket="adu-project-data",
    key="aisi-economy-index/pre_migration_backup/store_20260227.tar.gz",
)

Source: /lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store
Temp tar: /local/user/1483802411/hf_model_tar_wpzz8gh9/store_20260227.tar.gz
S3 target: s3://adu-project-data/aisi-economy-index/pre_migration_backup/store_20260227.tar.gz


In [2]:
from pathlib import Path
import boto3
from boto3.s3.transfer import TransferConfig


def upload_tar_gz_to_s3(
    tar_path,
    bucket,
    key,
    *,
    profile=None,
    verbose=True,
):
    tar_path = Path(tar_path).resolve()
    if not tar_path.exists():
        raise FileNotFoundError(f"tar file not found: {tar_path}")
    if not tar_path.is_file():
        raise ValueError(f"tar_path must be a file: {tar_path}")
    if not tar_path.name.endswith(".tar.gz"):
        raise ValueError("expected a .tar.gz file")

    if verbose:
        size_gb = tar_path.stat().st_size / 1024**3
        print("Local file:", tar_path)
        print(f"Size: {size_gb:.2f} GiB")
        print("S3 target:", f"s3://{bucket}/{key}")

    session = boto3.Session(profile_name=profile) if profile else boto3.Session()
    s3 = session.client("s3")

    # Multipart upload (safe for large files)
    config = TransferConfig(
        multipart_threshold=64 * 1024**2,
        multipart_chunksize=64 * 1024**2,
        max_concurrency=8,
        use_threads=True,
    )

    s3.upload_file(
        str(tar_path),
        bucket,
        key,
        Config=config,
    )

    if verbose:
        print("Upload complete")

    return str(tar_path)

In [ ]:
LLAMA_DIR = "/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store"

tar_and_upload_to_s3(
    source_dir=LLAMA_DIR,
    bucket="adu-project-data",
    key="aisi-economy-index/pre_migration_backup/store_20260227.tar.gz",
)

In [ ]:
upload_tar_gz_to_s3(
    "/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store_20260227.tar.gz",
    bucket="adu-project-data",
    key="aisi-economy-index/pre_migration_backup/store_20260227.tar.gz",
    profile=None,  # or your AWS profile name
)